### Build a Question Answering application over a Graph Database

In [2]:
## build a instance in neo4j and download the credentials.
## KnowledgeGraphs/Neo4j-2be841c0-Created-2026-04-05.txt

with open('/Users/subham/Desktop/GENAI/genai/48_KnowledgeGraphs/Neo4j-bc40e8f3-Created-2026-04-05.txt', 'r') as file:
    content = file.readlines()

for line in content:
    if line.startswith("NEO4J_URI="):
        NEO4J_URI = line.split("=", 1)[1].strip()
    elif line.startswith("NEO4J_USERNAME="):
        NEO4J_USERNAME = line.split("=", 1)[1].strip()
    elif line.startswith("NEO4J_PASSWORD="):
        NEO4J_PASSWORD = line.split("=", 1)[1].strip()

In [3]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [4]:
from neo4j import GraphDatabase

# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = NEO4J_URI
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)


with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")

Connection established.


In [31]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="bc40e8f3"  
)

In [6]:
## Dataset Moview 
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

In [32]:
print(moview_query)


LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))





In [33]:
graph.query(moview_query)

[]

In [34]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [25]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY_3")

In [26]:
from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11dbb3f20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11dbc2f60>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
cypher_prompt = ChatPromptTemplate.from_template("""
You are an expert Neo4j Cypher generator.

Given the schema:
{schema}

Generate a Cypher query for the question:
{question}

Return ONLY the Cypher query.
""")

cypher_chain = (
    cypher_prompt
    | llm
    | StrOutputParser()
)

In [38]:
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Given the query result:
{context}

Answer the question:
{question}
""")

answer_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

In [39]:
def run_query(question: str):
    # Get schema
    schema = graph.get_schema

    # Step 1: Generate Cypher
    cypher_query = cypher_chain.invoke({
        "schema": schema,
        "question": question
    })

    print("\nGenerated Cypher:")
    print(cypher_query)

    # Step 2: Execute query
    result = graph.query(cypher_query)

    print("\nRaw Result:")
    print(result)

    # Step 3: Generate final answer
    answer = answer_chain.invoke({
        "context": result,
        "question": question
    })

    return answer

In [40]:
response = run_query("Who was the director of the movie Casino")
print("\nFinal Answer:")
print(response)


Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name AS director;

Raw Result:
[{'director': 'Martin Scorsese'}]

Final Answer:
The director of the movie **Casino** was **Martin Scorsese**.


In [ ]:
response = run_query("Who were the actors of the movie Casino?")
print("\nFinal Answer:")
print(response)


Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Casino'})
RETURN p.name AS actor;

Raw Result:
[{'actor': 'Robert De Niro'}, {'actor': 'Joe Pesci'}, {'actor': 'Sharon Stone'}, {'actor': 'James Woods'}]

Final Answer:
The actors listed for the movie **Casino** are:

- Robert De Niro  
- Joe Pesci  
- Sharon Stone  
- James Woods


In [42]:
response = run_query("How many artists are there?")
print("\nFinal Answer:")
print(response)


Generated Cypher:
MATCH (p:Person)
WHERE (p)-[:ACTED_IN]-() OR (p)-[:DIRECTED]-()
RETURN count(DISTINCT p) AS artistCount;

Raw Result:
[{'artistCount': 1239}]

Final Answer:
There are **1,239 artists**.


In [43]:
response = run_query("How many movies has Tom Hanks acted in?")
print("\nFinal Answer:")
print(response)


Generated Cypher:
MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)
RETURN count(DISTINCT m) AS movieCount;

Raw Result:
[{'movieCount': 2}]

Final Answer:
Tom Hanks has acted in **2 movies**.
